# Stage 2 · YOLO ingredient detection → recipe vocabulary

This stage turns a **photo** into **ingredient names** the recommender understands:

1. a trained **YOLOv5** detector (`weights/best.pt`, 95 ingredient classes) finds ingredients in
   an image (`detect.py`);
2. each YOLO class label is **mapped** to the canonical ingredient token used by the recommender
   (`build_mapping.py` → `yolo_vocab_mapping.json`, applied by `yolo_mapping.py`).

The output feeds Stage 3 (`model/`) via `recommend(ingredients, df)`.

In [1]:
import os, sys
HERE = os.getcwd()
ROOT = os.path.dirname(HERE) if os.path.basename(HERE) == "yolo" else HERE
sys.path.insert(0, os.path.join(ROOT, "yolo"))
sys.path.insert(0, os.path.join(ROOT, "model", "src"))

import pandas as pd
import detect            # yolo/detect.py
import yolo_mapping      # yolo/yolo_mapping.py
import build_mapping     # yolo/build_mapping.py
import recipe_recommender as rr
print("paths set; ROOT =", ROOT)

paths set; ROOT = D:\Uchi_Ads_Hw\Bayesian\Final_project


## 1 · The detector — 95 ingredient classes

`best.pt` was trained with the ultralytics/yolov5 repo (cloned at `yolo/yolov5`). Loading handles
two portability details: a Linux-saved `PosixPath` (on Windows) and yolov5's noisy pip check.

In [2]:
names = detect.class_names()
print("number of YOLO classes:", len(names))
pd.DataFrame({"class_id": range(len(names)), "yolo_label": names}).head(20)

YOLOv5  70b964b Python-3.13.5 torch-2.12.0.dev20260314+cu128 CUDA:0 (NVIDIA GeForce RTX 5090 Laptop GPU, 24463MiB)



Fusing layers... 


Model summary: 212 layers, 21232788 parameters, 0 gradients, 49.1 GFLOPs


Adding AutoShape... 


number of YOLO classes: 95


,class_id,yolo_label
0,0,Romaine_lettuce
1,1,Spinach
2,2,Wakame
3,3,Chili_powder
4,4,Yellow_bell_pepper
5,5,Cooked_rice
6,6,Paprika
7,7,Vienna_sausage
8,8,Sausage
9,9,Tofu


## 2 · Map YOLO labels → recipe ingredients

For each class we (1) normalize it with the recommender's own canonicalizer, (2) accept it if it
occurs in the recipe vocabulary, else its head noun, else a rapidfuzz match — so a mapped label is
guaranteed to line up with the recommender's coverage set-intersection.

In [3]:
mapping = build_mapping.build()      # writes yolo_vocab_mapping.json and returns the dict
mapped = pd.DataFrame(sorted(mapping.items()), columns=["yolo_label", "ingredient"])
print(f"mapped {len(mapping)}/{len(names)} classes")
mapped.head(25)

[build_mapping] 95 YOLO classes vs 7056 recipe tokens
[build_mapping] mapped 93/95 labels -> D:\Uchi_Ads_Hw\Bayesian\Final_project\yolo\yolo_vocab_mapping.json
[build_mapping] unmapped: ['Gochujang', 'Jujube']
  Romaine_lettuce          -> romaine lettuce      [exact]
  Spinach                  -> spinach              [exact]
  Wakame                   -> wakame seaweed       [fuzzy(90)]
  Chili_powder             -> chili powder         [exact]
  Yellow_bell_pepper       -> yellow bell pepper   [exact]
  Cooked_rice              -> rice                 [exact]
  Paprika                  -> paprika              [exact]
  Vienna_sausage           -> vienna sausage       [exact]
  Sausage                  -> sausage              [exact]
  Tofu                     -> tofu                 [exact]
  Salsa                    -> salsa                [exact]
  Noodle                   -> noodle               [exact]
  Ramen                    -> raman noodle         [fuzzy(90)]
  Soybean_sprou

,yolo_label,ingredient
0,Almond,almond
1,Avocado,avocado
2,Bacon,bacon
3,Baguette,baguette
4,Banana,banana
5,Basil,basil
6,Bell_pepper,bell pepper
7,Bread,bread
8,Broccoli,broccoli
9,Butter,butter


## 3 · End-to-end on an image

On a real photo: `detect.detect("path/to/photo.jpg")`. Here we use a **mock** detection set (the
notebook ships without a sample photo) to show the full chain detect → map → recommend. Swap in a
real path to try your own image, or run `python run_pipeline.py --image photo.jpg`.

In [4]:
# detections = detect.detect("path/to/your_photo.jpg", conf=0.25)   # <- real usage
detections = [("Chicken", 0.93), ("Tomato", 0.88), ("Garlic", 0.81),
              ("Onion", 0.79), ("Cheese", 0.60), ("Basil", 0.55)]            # mock
ingredients = yolo_mapping.map_labels(detections)
print("detected :", [d[0] for d in detections])
print("mapped   :", ingredients)

df = pd.read_csv(os.path.join(ROOT, "data", "recipes_clean.csv"))
model = rr.load_model(os.path.join(ROOT, "model", "models", "lda_model.pkl"))
rr._STATE = {"model": model, "df_id": id(df)}
recs = rr.recommend(ingredients, df, top_n=5)
pd.DataFrame(recs)[["recipe_name", "score", "coverage", "predicted_rating"]]

detected : ['Chicken', 'Tomato', 'Garlic', 'Onion', 'Cheese', 'Basil']
mapped   : ['chicken', 'tomato', 'garlic', 'onion', 'cheese', 'basil']


[filter_candidates] threshold=0.5: 35 candidate recipes


,recipe_name,score,coverage,predicted_rating
0,braised balsamic chicken with garlic and onions,0.5952,3/5 ingredients,4.576
1,zucchini packets for the grill,0.5057,4/8 ingredients,4.630
2,garlicky tomato bruschetta,0.4224,3/6 ingredients,4.634
3,simple roasted tomato and garlic sauce,0.4147,2/3 ingredients,3.840
4,healthy cucumber tomato salad,0.4102,3/6 ingredients,4.502


## Summary

- **Detect:** `detect.detect(image)` → ingredient labels (YOLOv5, 95 classes).
- **Map:** `yolo_vocab_mapping.json` aligns YOLO labels to recipe-vocabulary ingredients.
- **Recommend:** the mapped ingredients go straight into Stage 3's `recommend()`.
- One command for the whole chain: `python run_pipeline.py --image photo.jpg`.